# Lab 04 Lab 04: Data Unions and Joins Pipeline

Santiago Elí Jiménez Aguilar 

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Lab 04 - Data Unions and Joins"

su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 00:37:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkUtils: Lab 04 - Data Unions and Joins>

In [2]:
#Schemas

agencies_schema = SparkUtils.generate_schema([
    ("agency_id",   "int"),
    ("agency_info", "string"),
])

brands_schema = SparkUtils.generate_schema([
    ("brand_id",   "int"),
    ("brand_info", "string"),
])

cars_schema = SparkUtils.generate_schema([
    ("car_id",   "int"),
    ("car_info", "string"),
])

customers_schema = SparkUtils.generate_schema([
    ("customer_id",   "int"),
    ("customer_info", "string"),
])

rentals_schema = SparkUtils.generate_schema([
    ("rental_id",   "int"),
    ("rental_info", "string"),
])

In [3]:
#Read csv

agencies_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(agencies_schema) \
                .csv("/opt/spark/work-dir/data/car_service/agencies")

brands_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(brands_schema) \
                .csv("/opt/spark/work-dir/data/car_service/brands")

cars_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(cars_schema) \
                .csv("/opt/spark/work-dir/data/car_service/cars")

customers_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(customers_schema) \
                .csv("/opt/spark/work-dir/data/car_service/customers")

rentals_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(rentals_schema) \
                .csv("/opt/spark/work-dir/data/car_service/rentals")

In [4]:


from pyspark.sql.functions import get_json_object, col

# Agencies -> agency_name
agencies_parsed = agencies_df.select(
    col("agency_id"),
    get_json_object(col("agency_info"), "$.agency_name").alias("agency_name")
)

# Cars -> car_name
cars_parsed = cars_df.select(
    col("car_id"),
    get_json_object(col("car_info"), "$.car_name").alias("car_name")
)

# Customers -> customer_name
customers_parsed = customers_df.select(
    col("customer_id"),
    get_json_object(col("customer_info"), "$.customer_name").alias("customer_name")
)

# Rentals -> extract foreign keys from JSON
rentals_parsed = rentals_df.select(
    col("rental_id"),
    get_json_object(col("rental_info"), "$.car_id").cast("int").alias("car_id"),
    get_json_object(col("rental_info"), "$.customer_id").cast("int").alias("customer_id"),
    get_json_object(col("rental_info"), "$.agency_id").cast("int").alias("agency_id")
)

print("Transformations defined lazily — still no action triggered.")

Transformations defined lazily — still no action triggered.


In [5]:
(
    rentals_parsed
    # Join with Cars -> car_name
    .join(cars_parsed, on="car_id", how="inner")
    # Join with Agencies -> agency_name
    .join(agencies_parsed, on="agency_id", how="inner")
    # Join with Customers -> customer_name
    .join(customers_parsed, on="customer_id", how="inner")
    # Keep only the required columns
    .select("rental_id", "car_name", "agency_name", "customer_name")
    # Single Action
    .show()
)

+---------+--------------------+-------------+---------------+
|rental_id|            car_name|  agency_name|  customer_name|
+---------+--------------------+-------------+---------------+
|    11891|Wallace-Carlson M...|  NYC Rentals| Margaret Jones|
|    11892|Grimes-Green Model 8|LA Car Rental|Albert Williams|
|    11893|Stewart-Allen Mod...|      SF Cars|  Caleb Fleming|
|    11894|  Campos PLC Model 4|  NYC Rentals|  Andrew Butler|
|    11895|  Wagner LLC Model 1|      SF Cars|  Kristin Potts|
|    11896|Jones, Jefferson ...|LA Car Rental|   Jeremy Parks|
|    11897|Lopez and Sons Mo...| Zapopan Auto|    Terry Wells|
|    11898| Salazar Ltd Model 8|      SF Cars|  Marc Williams|
|    11899|Villanueva PLC Mo...|LA Car Rental| Danny Williams|
|    11900|Faulkner-Howard M...|      SF Cars| Eric Owens PhD|
|    11901|Faulkner-Howard M...|  NYC Rentals|    Laura Perry|
|    11902|Faulkner-Howard M...|  NYC Rentals|     Paul Brown|
|    11903|Atkinson Ltd Mode...| Zapopan Auto|Alexa Her

In [6]:
su.spark_context().stop()